# Extração de Parágrafos Marcados - Processamento de Documentos Jurídicos
Notebook para extrair apenas parágrafos com marcadores de soressão, sem incluir parágrafos subsequentes, e salvar em JSON no Google Drive.

## Instalar e Importar Bibliotecas

In [ ]:
%pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 29.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=bdce4d7e3100273a10dfc2c701966f5ca933d182bb0d8bbb39ed373335b87c67
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
import json
import os
import re
import subprocess
import nltk
import spacy

from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from langdetect import detect, LangDetectException
from google.colab import drive
from tqdm import tqdm
from pathlib import Path

## Baixar Plugins e Configurar NLP

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
nlp = spacy.load('en_core_web_sm')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## Montar Google Drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


## Definição das Classes de Processamento

In [ ]:
@dataclass
class ProcessedParagraph:
    """Estrutura para armazenar parágrafo processado"""
    doc_id: str
    paragraph_id: str
    original_text: str
    cleaned_text: str
    tokens: List[str]
    processed_tokens: List[str]
    has_suppressed_marker: bool
    marker_types: List[str]
    is_french: bool


class LegalDocumentPreprocessor:
    """
    Pré-processador especializado para documentos jurídicos.
    Extrai e processa parágrafos com marcadores especiais.
    """

    def __init__(self, use_spacy: bool = True):
        """
        Args:
            use_spacy: Se True, usa spaCy para tokenização e lematização.
                      Se False, usa NLTK com stemming.
        """
        self.use_spacy = use_spacy
        self.stemmer = PorterStemmer()
        self.stop_words = set(stopwords.words('english'))

        # Demarcadores especiais a buscar
        self.suppressed_markers = [
            'FRAGMENT_SUPPRESSED',
            'FRAGMENT_SUPRESSED',
            'CITATION_SUPPRESSED',
            'CITATION_SUPRESSED',
            'REFERENCE_SUPPRESSED',
            'REFERENCE_SUPRESSED',
        ]

        # Padrão para identificar números de parágrafos
        self.paragraph_pattern = r'\[(\d{1,4})\]'

    def is_french(self, text: str) -> bool:
        """Detecta se o texto está em francês"""
        if not text or len(text.strip()) < 10:
            return False
        try:
            lang = detect(text)
            return lang == 'fr'
        except LangDetectException:
            return False

    def _normalize_text(self, document_text: str) -> str:
        processed = document_text.replace('\n', ' ').replace('•', ' ')
        processed = re.sub(r'\s+', ' ', processed).strip()
        return processed

    def extract_all_paragraphs(self, document_text: str) -> List[Tuple[str, str]]:
        """Extrai todos os parágrafos do documento."""
        processed = self._normalize_text(document_text)
        paragraph_matches = list(re.finditer(self.paragraph_pattern, processed))
        if not paragraph_matches:
            return []

        results: List[Tuple[str, str]] = []
        for i, match in enumerate(paragraph_matches):
            para_id = match.group(1)
            start = match.start()
            if i + 1 < len(paragraph_matches):
                end = paragraph_matches[i + 1].start()
            else:
                end = len(processed)
            para_text = processed[start:end].strip()
            results.append((para_id, para_text))
        return results

    def extract_paragraphs_with_markers_only(self, document_text: str) -> List[Tuple[str, str]]:
        """
        Extrai APENAS os parágrafos que contêm demarcadores (SEM incluir subsequentes).

        Args:
            document_text: Texto completo do documento

        Returns:
            Lista de tuplas (id_parágrafo, texto_parágrafo)
        """
        processed = self._normalize_text(document_text)
        paragraph_matches = list(re.finditer(self.paragraph_pattern, processed))

        if not paragraph_matches:
            return []

        results = []
        markers_upper = [m.upper() for m in self.suppressed_markers]

        for i, match in enumerate(paragraph_matches):
            para_id = match.group(1)
            start = match.start()

            if i + 1 < len(paragraph_matches):
                end = paragraph_matches[i + 1].start()
            else:
                end = len(processed)

            para_text = processed[start:end].strip()
            para_upper = para_text.upper()
            has_marker = any(marker in para_upper for marker in markers_upper)

            if has_marker:
                # Aqui apenas adicionamos o próprio parágrafo, SEM o subsequente
                results.append((para_id, para_text))

        return results

    def clean_paragraph(self, text: str) -> str:
        """Remove demarcadores e formatação desnecessária."""
        for marker in self.suppressed_markers:
            text = re.sub(rf'<{marker}>', ' ', text, flags=re.IGNORECASE)
            text = re.sub(marker, ' ', text, flags=re.IGNORECASE)

        text = re.sub(self.paragraph_pattern, ' ', text)
        text = re.sub(r'[<>]', ' ', text)
        text = re.sub(r'\[End of document\]', ' ', text, flags=re.IGNORECASE)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def tokenize_and_process(self, text: str) -> Tuple[List[str], List[str]]:
        """Tokeniza e processa o texto."""
        if self.use_spacy:
            doc = nlp(text.lower())

            tokens = [
                token.text for token in doc
                if token.is_alpha and
                   token.text not in self.stop_words and
                   len(token.text) > 2
            ]

            lemmatized = [
                token.lemma_ for token in doc
                if token.is_alpha and
                   token.lemma_ not in self.stop_words and
                   len(token.lemma_) > 2
            ]

            return tokens, lemmatized
        else:
            tokens = word_tokenize(text.lower())
            tokens = [
                token for token in tokens
                if token.isalpha() and
                   token not in self.stop_words and
                   len(token) > 2
            ]
            stemmed = [self.stemmer.stem(token) for token in tokens]
            return tokens, stemmed

    def identify_markers(self, text: str) -> List[str]:
        """Identifica quais demarcadores estão presentes no texto."""
        found_markers = []
        upper_text = text.upper()
        for marker in self.suppressed_markers:
            if marker.upper() in upper_text or f'<{marker}>' in upper_text:
                found_markers.append(marker)
        return found_markers

    def process_document(self, doc_id: str, document_text: str, filter_french: bool = True) -> List[ProcessedParagraph]:
        """
        Processa o documento completo, extraindo APENAS parágrafos com marcadores.

        Args:
            doc_id: ID do documento
            document_text: Texto completo do documento
            filter_french: Se True, filtra parágrafos em francês

        Returns:
            Lista de parágrafos processados
        """
        # Extrai apenas parágrafos com marcadores (sem subsequentes)
        paragraphs_with_markers = self.extract_paragraphs_with_markers_only(document_text)

        processed_paragraphs = []

        for para_id, para_text in paragraphs_with_markers:
            is_french_text = self.is_french(para_text)
            if filter_french and is_french_text:
                continue

            cleaned = self.clean_paragraph(para_text)
            if not cleaned or len(cleaned) < 10:
                continue

            tokens, processed_tokens = self.tokenize_and_process(cleaned)
            markers = self.identify_markers(para_text)
            has_marker = len(markers) > 0

            processed_paragraphs.append(ProcessedParagraph(
                doc_id=doc_id,
                paragraph_id=para_id,
                original_text=para_text,
                cleaned_text=cleaned,
                tokens=tokens,
                processed_tokens=processed_tokens,
                has_suppressed_marker=has_marker,
                marker_types=markers,
                is_french=is_french_text
            ))

        return processed_paragraphs

In [ ]:
class QueryDocumentProcessor:
    """Processa apenas documentos query a partir do arquivo de labels."""

    def __init__(self, preprocessor: LegalDocumentPreprocessor):
        self.preprocessor = preprocessor
        self.query_paragraphs: List[ProcessedParagraph] = []
        self.processed_documents = 0
        self.skipped_documents = 0
        self.errors = 0

    def _serialize_paragraphs(self, paragraphs: List[ProcessedParagraph]) -> List[Dict]:
        """Serializa parágrafos para JSON."""
        serialized = []
        for para in paragraphs:
            serialized.append({
                'doc_id': para.doc_id,
                'paragraph_id': para.paragraph_id,
                'original_text': para.original_text,
                'cleaned_text': para.cleaned_text,
                'tokens': para.tokens,
                'processed_tokens': para.processed_tokens,
                'has_suppressed_marker': para.has_suppressed_marker,
                'marker_types': para.marker_types,
                'is_french': para.is_french,
            })
        return serialized

    def _normalize_doc_id(self, doc_id: str) -> str:
        """Normaliza ID do documento."""
        doc_id = doc_id.strip()
        if doc_id.lower().endswith('.txt'):
            return doc_id[:-4]
        return doc_id

    def process_query_documents(
        self,
        documents_folder: str,
        labels_file: str,
        max_docs: Optional[int] = None,
        verbose: bool = True,
        filter_french: bool = True,
    ) -> List[ProcessedParagraph]:
        """Processa apenas documentos de query, extraindo parágrafos com marcador."""
        with open(labels_file, 'r', encoding='utf-8') as f:
            labels = json.load(f)

        query_ids = {self._normalize_doc_id(k) for k in labels.keys()}

        if verbose:
            print(f"Query IDs encontrados: {len(query_ids)}")

        folder = Path(documents_folder)
        if not folder.exists():
            raise ValueError(f"Pasta '{documents_folder}' não encontrada.")

        files = sorted(list(folder.glob('*.txt')))
        if max_docs:
            files = files[:max_docs]

        if verbose:
            print(f"Arquivos totais disponíveis: {len(files)}")
            print("Processando somente arquivos de query...")

        iterator = tqdm(files, desc="Processando queries") if verbose else files

        self.query_paragraphs = []
        self.processed_documents = 0
        self.skipped_documents = 0
        self.errors = 0

        for file_path in iterator:
            doc_id = file_path.stem
            doc_id_norm = self._normalize_doc_id(doc_id)

            if doc_id_norm not in query_ids:
                self.skipped_documents += 1
                continue

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()

                paragraphs = self.preprocessor.process_document(
                    doc_id,
                    content,
                    filter_french=filter_french,
                )
                self.query_paragraphs.extend(paragraphs)
                self.processed_documents += 1
            except Exception as e:
                self.errors += 1
                if verbose:
                    print(f"Erro ao processar {file_path}: {e}")

        return self.query_paragraphs

    def save_query_to_json(
        self,
        output_dir: str,
        filename: str = 'query_marked_paragraphs.json',
        verbose: bool = True,
    ) -> str:
        """Salva os query paragraphs processados em um único arquivo JSON."""
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)

        query_serialized = self._serialize_paragraphs(self.query_paragraphs)
        query_file = output_path / filename

        with open(query_file, 'w', encoding='utf-8') as f:
            json.dump(query_serialized, f, ensure_ascii=False, indent=2)

        if verbose:
            print(f"Query parágrafos salvos em: {query_file}")

        return str(query_file)

## Configurar Caminhos e Parâmetros

In [ ]:
# Configurar caminhos - AJUSTE CONFORME SEU AMBIENTE
BASE_PATH = '/content/drive/MyDrive/tcc/data/'

# Caminhos de entrada
TRAIN_FILES = os.path.join(BASE_PATH, 'task1_train_files_2025/task2025_train')
TRAIN_LABELS_FILE = os.path.join(BASE_PATH, 'task1_train_labels_2025.json')

# Parâmetros de processamento
MAX_DOCUMENTS = None  # None para processar todos, ou um número inteiro para limitar
FILTER_FRENCH = True  # Filtrar parágrafos em francês
USE_SPACY = True  # Usar spaCy para processamento (True) ou NLTK (False)

print("Caminhos configurados:")
print(f"  Base: {BASE_PATH}")
print(f"  Documentos: {TRAIN_FILES}")
print(f"  Labels: {TRAIN_LABELS_FILE}")

Caminhos configurados:
  Base: /content/drive/MyDrive/tcc/data/
  Documentos: /content/drive/MyDrive/tcc/data/task1_train_files_2025/task2025_train
  Labels: /content/drive/MyDrive/tcc/data/task1_train_labels_2025.json


## Processar Documentos e Extrair Parágrafos Marcados

In [ ]:
# Inicializar o pré-processador
print("Inicializando pré-processador...")
preprocessor = LegalDocumentPreprocessor(use_spacy=USE_SPACY)
processor = QueryDocumentProcessor(preprocessor)
print("✓ Pré-processador inicializado")

Inicializando pré-processador...
✓ Pré-processador inicializado


In [ ]:
# Processar apenas documentos query
print("\nIniciando processamento dos documentos query...\n")

query_paragraphs = processor.process_query_documents(
    documents_folder=TRAIN_FILES,
    labels_file=TRAIN_LABELS_FILE,
    max_docs=MAX_DOCUMENTS,
    verbose=True,
    filter_french=FILTER_FRENCH,
)

print("\nProcessamento concluído!")
print(f"  - Documentos query processados: {processor.processed_documents}")
print(f"  - Arquivos ignorados (não query): {processor.skipped_documents}")
print(f"  - Erros: {processor.errors}")
print(f"  - Query parágrafos extraídos: {len(query_paragraphs)}")


Iniciando processamento dos documentos query...

Query IDs encontrados: 1678
Arquivos totais disponíveis: 7350
Processando somente arquivos de query...


Processando queries: 100%|██████████| 7350/7350 [10:35<00:00, 11.56it/s]


Processamento concluído!
  - Documentos query processados: 1678
  - Arquivos ignorados (não query): 5672
  - Erros: 0
  - Query parágrafos extraídos: 19442


## Definir Caminho de Saída no Drive
Especifique o caminho no Google Drive onde deseja salvar os arquivos JSON processados.

In [ ]:
# ===== CONFIGURE O CAMINHO DE SAÍDA AQUI =====
# Defina o caminho completo no Google Drive onde deseja salvar os JSONs
# Exemplo: '/content/drive/MyDrive/tcc/data/marked_paragraphs/'

OUTPUT_DIR = os.path.join(BASE_PATH, 'bm25/bm25-marked-paragraphs-to-llm/cache')

# Se você quiser usar um caminho diferente, descomente e edite a linha abaixo:
# OUTPUT_DIR = '/content/drive/MyDrive/seu/caminho/aqui/'

print(f"Caminho de saída configurado: {OUTPUT_DIR}")

Caminho de saída configurado: /content/drive/MyDrive/tcc/data/bm25/bm25-marked-paragraphs-to-llm/cache


## Salvar Resultados em JSON

In [ ]:
print("Salvando resultados em JSON...\n")

# Salvar apenas o arquivo de query paragraphs
query_file = processor.save_query_to_json(
    output_dir=OUTPUT_DIR,
    verbose=True,
)

print("\n" + "=" * 70)
print("RESUMO DO PROCESSAMENTO")
print("=" * 70)

Salvando resultados em JSON...

Query parágrafos salvos em: /content/drive/MyDrive/tcc/data/bm25/bm25-marked-paragraphs-to-llm/cache/query_marked_paragraphs.json

RESUMO DO PROCESSAMENTO
